# Disposición inadecuada de basura y percepción de inseguridad

El índice usado para comparar el dataset Disposición inadecuada de basura es `brecha_percepcion = (percepción de inseguridad nocturna) - (crímenes estimados por 100.000 habitantes)`. Los crimenes estimados incorporan los delitos de alto impacto registrados / porcentaje de hogares que afirman haber decnunciado / 100.000 habitantes. La inseguridad nocturna y la disposición inadecuada se expresan como porcentajes de hogares, respectivamente, en una escala de 0 a 100. El indice final representa la posición relativa de cada localidad frente al promedio. Un valor positivo significa que la percepción supera lo que sugieren los delitos estimados.

## Datasets usados


El archivo `delitos_alto_impacto.geojson` aporta los delitos de alto impacto registrados por localidad en 2025.

La `encuesta_distrital_percepcion_2025.xlsx` aporta la denuncia de delitos y la inseguridad percibida al caminar de noche.

Las `proyecciones_poblacion_localidad_2005_2035.ods` permiten expresar el crimen por 100.000 habitantes. El archivo `disposicion_inadecuada_de_basura.csv` es la exposición ambiental que se contrasta con el índice.

Esta primera celda prepara las herramientas de lectura, limpieza, análisis estadístico y visualización. También define las rutas de trabajo y una función para homogeneizar los nombres de las localidades, porque los archivos pueden usar tildes, espacios no separables o pequeñas variaciones de escritura.

In [74]:
from pathlib import Path
import json, re, unicodedata, zipfile, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

ROOT = Path.cwd()
if not (ROOT/'data'/'raw').exists() and (ROOT.parent/'data'/'raw').exists(): ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'; OUT = ROOT / 'outputs'; PROC = ROOT / 'data' / 'processed'
OUT.mkdir(exist_ok=True); PROC.mkdir(exist_ok=True)
def clean(x): return re.sub(r'\s+', ' ', str(x or '')).strip()
def key(x): return re.sub(r'[^A-Z0-9]', '', unicodedata.normalize('NFKD', clean(x)).encode('ascii','ignore').decode().upper())
LOC = {'USAQUEN':'Usaquen','CHAPINERO':'Chapinero','SANTAFE':'Santa Fe','SANCRISTOBAL':'San Cristobal','USME':'Usme','TUNJUELITO':'Tunjuelito','BOSA':'Bosa','KENNEDY':'Kennedy','FONTIBON':'Fontibon','ENGATIVA':'Engativa','SUBA':'Suba','BARRIOSUNIDOS':'Barrios Unidos','TEUSAQUILLO':'Teusaquillo','LOSMARTIRES':'Los Martires','ANTONIONARINO':'Antonio Narino','PUENTEARANDA':'Puente Aranda','LACANDELARIA':'La Candelaria','CANDELARIA':'La Candelaria','RAFAELURIBEURIBE':'Rafael Uribe Uribe','CIUDADBOLIVAR':'Ciudad Bolivar','SUMAPAZ':'Sumapaz'}
locality = lambda x: LOC.get(key(x), clean(x).title())

Aquí se abre el dataset Delitos de Alto Impacto en formato GeoJSON. Se seleccionan los campos de 2025 y se suman los conteos de los delitos incluidos para obtener, por localidad, el número administrativo de hechos registrados. Esta suma todavía no corrige la no denuncia; por eso se denomina conteo oficial.

In [75]:
# 1. Delitos: se lee directamente el GeoJSON descargado del catálogo.
with open(RAW/'delitos_alto_impacto.geojson', encoding='utf-8-sig') as f:
    crime = json.load(f)
crime_rows = pd.DataFrame([f['properties'] for f in crime['features']])
crime_rows['locality'] = crime_rows['CMNOMLOCAL'].map(locality)
# Campos 2025 publicados por DAILoc: homicidios, lesiones, hurto a personas,
# hurto de vehículos, bicicletas, residencias, comercio y violencia intrafamiliar.
crime_fields = ['CMH25CONT','CMLP25CONT','CMHP25CONT','CMHR25CONT','CMHA25CONT','CMHB25CONT','CMHM25CONT','CMVI25CONT']
crime_rows['official_crimes_2025'] = crime_rows[crime_fields].fillna(0).sum(axis=1)
crime_rows[['locality','official_crimes_2025']].head()

,locality,official_crimes_2025
0,Fontibon,9743.0
1,Ciudad Bolivar,10364.0
2,Los Martires,6145.0
3,Antonio Narino,2873.0
4,Chapinero,9653.0


### Denuncia y crimen estimado

La encuesta pregunta, por tipo de delito, si el hogar denunció. Cada conteo administrativo se corrige con la proporción de denuncia del mismo tipo: `crímenes estimados = delitos registrados / proporción que denunció`. Se limita la proporción a 10%–95% para evitar explosiones por celdas pequeñas.

Esta celda lee la Encuesta Distrital de Percepción 2025. Del Cuadro 16A se extraen, por localidad y delito, los hogares que denunciaron y los que no denunciaron. La proporción de denuncia es una fracción entre 0 y 1; al dividir el conteo administrativo por esa fracción se obtiene una aproximación del número de hechos que pudieron ocurrir incluyendo los no denunciados.

In [76]:
from openpyxl import load_workbook
SURVEY = RAW/'encuesta_distrital_percepcion_2025.xlsx'
def read_sheet(name):
    wb=load_workbook(SURVEY,read_only=True,data_only=True,keep_links=False); ws=wb[name]; rows=[]
    for r in ws.iter_rows(min_row=15,max_col=11,values_only=True):
        if r[0] is not None: rows.append(r)
    wb.close(); d=pd.DataFrame(rows,columns=['locality','question','category','people','p_cv','people_pct','p_cv_pct','households','h_cv','households_pct','h_cv_pct'])
    for c in ['locality','question','category']: d[c]=d[c].map(clean)
    return d[d.locality.map(key).isin(LOC)].copy()
d16=read_sheet('Cuadro 16A'); d16['locality']=d16.locality.map(locality)
d16['crime_type']=d16.question.str.replace(r'^\s*[a-z]\.?\s*','',regex=True,case=False)
report=[]
for (loc,typ),g in d16.groupby(['locality','crime_type']):
    by={str(r.category):r for r in g.itertuples()}; total=float(getattr(by.get('Total'),'households',0) or 0); yes=float(getattr(by.get('Sí'),'households',0) or 0)
    if total: report.append({'locality':loc,'crime_type':typ,'report_share':yes/total})
report=pd.DataFrame(report); report['report_share_used']=report.report_share.clip(.10,.95)
# Cada tipo de delito se corrige con su propia proporción de denuncia.
mapping=[('person_theft','Hurto a personas',['CMHP25CONT']),('vehicle_theft','Hurto de vehículos',['CMHA25CONT','CMHM25CONT']),('bicycle_theft','Hurto de bicicleta',['CMHB25CONT']),('homicide','Homicidios',['CMH25CONT']),('residential_theft','Hurto a residencias',['CMHR25CONT']),('personal_injury','Lesiones personales',['CMLP25CONT']),('domestic_violence','Violencia intrafamiliar',['CMVI25CONT'])]
estimated=crime_rows[['locality']].copy(); estimated['estimated_crimes_2025']=0.0
for label,prefix,fields in mapping:
    shares=report[report.crime_type.str.startswith(prefix)].groupby('locality').report_share_used.mean().to_dict()
    tmp=crime_rows[['locality']+fields].copy(); tmp['official_count']=tmp[fields].fillna(0).sum(axis=1); tmp['report_share_used']=tmp.locality.map(shares); tmp['estimated_count']=tmp.official_count/tmp.report_share_used
    estimated['estimated_crimes_2025'] += tmp.estimated_count.fillna(0)
base=crime_rows[['locality','official_crimes_2025']].merge(estimated,on='locality',how='left')
base[['locality','official_crimes_2025','estimated_crimes_2025']].head()

,locality,official_crimes_2025,estimated_crimes_2025
0,Fontibon,9743.0,22296.667088
1,Ciudad Bolivar,10364.0,22048.788059
2,Los Martires,6145.0,23527.149533
3,Antonio Narino,2873.0,5806.522949
4,Chapinero,9653.0,11208.900995


Ahora se incorporan las proyecciones de población y la percepción de seguridad. La población de 2025 convierte los crímenes estimados en una tasa comparable por cada 100.000 habitantes. Del Cuadro 19A de la encuesta se suman las respuestas Inseguro/a y Muy inseguro para obtener un porcentaje de personas, entre 0 y 100, que siente inseguridad al caminar de noche.

In [77]:
# 2. Población y percepción nocturna.
# El ODS se lee directamente como XML para no depender de un motor externo.
TNS='urn:oasis:names:tc:opendocument:xmlns:table:1.0'; ONS='urn:oasis:names:tc:opendocument:xmlns:office:1.0'
def ods_row_values(el,limit=314):
    values=[]
    for cell in list(el):
        if not cell.tag.endswith('table-cell'): continue
        rep=int(cell.attrib.get(f'{{{TNS}}}number-columns-repeated','1')); text=' '.join(''.join(cell.itertext()).split()); val=cell.attrib.get(f'{{{ONS}}}value',text); values.extend([val]*min(rep,max(0,limit-len(values))))
        if len(values)>=limit: break
    return values
pop_rows=[]
with zipfile.ZipFile(RAW/'proyecciones_poblacion_localidad_2005_2035.ods') as z, z.open('content.xml') as f:
    for _,el in ET.iterparse(f,events=('end',)):
        if el.tag==f'{{{TNS}}}table-row':
            vals=ods_row_values(el)
            if len(vals)>313 and clean(vals[7])=='2025': pop_rows.append({'locality':locality(vals[5]),'population_2025':float(vals[313])})
            el.clear()
pop=pd.DataFrame(pop_rows).groupby('locality',as_index=False)['population_2025'].sum(); pop.head()

,locality,population_2025
0,Antonio Narino,79758.0
1,Barrios Unidos,139756.0
2,Bosa,767340.0
3,Chapinero,163693.0
4,Ciudad Bolivar,680206.0


En la hoja de población se debe seleccionar la columna 2025 y normalizar el nombre de localidad. La siguiente celda deja explícita la selección; si el portal cambia el encabezado, basta cambiar `pop_col`.

Esta selección identifica las columnas de localidad y población 2025 en la hoja de proyecciones. Después se calcula la tasa de crimen estimado por 100.000 habitantes y se une el porcentaje de inseguridad nocturna a la tabla territorial.

In [78]:
pop2 = pop.copy(); pop2['locality']=pop2.locality.map(locality)
base=base.merge(pop2,on='locality',how='left'); base['crime_rate_100k']=100000*base.estimated_crimes_2025/base.population_2025
d20=read_sheet('Cuadro 19A'); d20['locality']=d20.locality.map(locality)
# Categorías de inseguridad nocturna; se suman Muy inseguro e Inseguro/a.
night=d20[d20.category.isin(['Muy inseguro','Inseguro/a'])].groupby('locality',as_index=False).people_pct.sum().rename(columns={'people_pct':'insecurity_noche_pct'})
base=base.merge(night,on='locality',how='left')

En este punto se construye el índice central. Se estandarizan la tasa de crimen y el porcentaje de inseguridad mediante puntuaciones z, que no tienen unidades y expresan cuántas desviaciones estándar se encuentra cada localidad del promedio. Luego se incorpora el dataset Disposición inadecuada de basura; su variable principal es el porcentaje de hogares, entre 0 y 100, que reporta esa condición.

In [79]:
# 3. Índice único y disposición inadecuada de basura.
lr=np.log1p(base.crime_rate_100k); base['crime_z']=(lr-lr.mean())/lr.std(ddof=0); base['insecurity_z']=(base.insecurity_noche_pct-base.insecurity_noche_pct.mean())/base.insecurity_noche_pct.std(ddof=0)
base['perception_excess_index']=base.insecurity_z-base.crime_z
bot=pd.read_csv(PROC/'botaderos_inadecuados-procesado.csv'); bot['locality']=bot.Loc.map(locality); bot['disposicion_inadecuada_pct']=100*bot['Porcentaje.Si']
final=base.merge(bot[['locality','disposicion_inadecuada_pct']],on='locality',how='left').dropna(subset=['perception_excess_index','disposicion_inadecuada_pct'])
rho,p=spearmanr(final.disposicion_inadecuada_pct,final.perception_excess_index); r,pp=pearsonr(final.disposicion_inadecuada_pct,final.perception_excess_index)
print({'n':len(final),'pearson_r':r,'pearson_p':pp,'spearman_rho':rho,'spearman_p':p})
final.to_csv(PROC/'indice_localidad_disposicion_inadecuada.csv',index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\cesar\\OneDrive\\Documents\\GitHub\\datajam-udata\\data\\processed\\botaderos_inadecuados-procesado.csv'

La última celda genera la figura para el pitch. Cada punto representa una localidad: el eje horizontal muestra el porcentaje de hogares con disposición inadecuada y el eje vertical muestra el exceso de inseguridad percibida. La línea resume la tendencia lineal y los coeficientes Pearson y Spearman cuantifican la relación, sin presentarla como causalidad.

In [ ]:
# 4. Output principal: comparación de localidades.
fig,ax=plt.subplots(figsize=(11,7.5)); ax.scatter(final.disposicion_inadecuada_pct,final.perception_excess_index,c=np.where(final.perception_excess_index>=0,'#D1495B','#2878B5'),s=90,edgecolor='white')
for x in final.itertuples(): ax.annotate(x.locality,(x.disposicion_inadecuada_pct,x.perception_excess_index),xytext=(4,3),textcoords='offset points',fontsize=7)
coef=np.polyfit(final.disposicion_inadecuada_pct,final.perception_excess_index,1); xs=np.linspace(final.disposicion_inadecuada_pct.min(),final.disposicion_inadecuada_pct.max(),100); ax.plot(xs,np.polyval(coef,xs),color='#333')
ax.axhline(0,color='#777',ls='--'); ax.set_xlabel('Hogares con disposición inadecuada de basura (%)'); ax.set_ylabel('Exceso perceptual: inseguridad − crimen estimado'); ax.set_title('Disposición inadecuada de basura y percepción de inseguridad',fontweight='bold'); ax.text(.01,.98,f'Correlación Pearson: r={r:.2f} | Correlación Spearman: ρ={rho:.2f}',transform=ax.transAxes,va='top'); ax.grid(alpha=.2); fig.savefig(OUT/'disposicion_inadecuada_vs_exceso_perceptual.png',dpi=190,bbox_inches='tight'); plt.show()

## Lectura para el pitch

Si la asociación es positiva, la disposición inadecuada puede funcionar como una señal visible de deterioro urbano asociada con mayor temor. Eso permitiría priorizar limpieza, recolección y recuperación del espacio público en las localidades donde la percepción supera el crimen estimado. Es una relación exploratoria y debe validarse después a escala de barrio.